# Description

Reads the 490 drug-disease prediction HDF5 files from `011-prediction-*` notebooks and computes
final performance measures (AUROC, average precision) against the PharmacotherapyDB gold standard.

Mirrors `phenoplier/nbs/30_drug_disease_associations/100-lincs/021-prediction_performance.ipynb` exactly.

Aggregation (same as PhenoPlier):
1. Group by (trait, drug, method, tissue) → **average ranks across thresholds**.
2. Group by (trait, drug, method) → **max across tissues**.

Files expected: 2 methods × 49 tissues × 5 thresholds = 490 HDF5 files.

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from collections import defaultdict
from IPython.display import display

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

from pyprojroot import here

# Settings

In [3]:
N_TISSUES = 49
N_THRESHOLDS = 5
N_PREDICTION_FILES_PER_METHOD = N_TISSUES * N_THRESHOLDS  # 245
N_PREDICTION_FILES_TOTAL = 2 * N_PREDICTION_FILES_PER_METHOD  # 490

In [4]:
DATA_DIR = here('data/archs4/drug_diseases_associations')
display(DATA_DIR)
assert DATA_DIR.exists()

OUTPUT_DIR = here('output/drug_disease_analyses') / 'lincs'
display(OUTPUT_DIR)
assert OUTPUT_DIR.exists()

OUTPUT_PREDICTIONS_DIR = OUTPUT_DIR / 'predictions' / 'dotprod_neg'
display(OUTPUT_PREDICTIONS_DIR)
assert OUTPUT_PREDICTIONS_DIR.exists()

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/archs4/drug_diseases_associations')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg')

# Load PharmacotherapyDB gold standard

In [5]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
display(gold_standard.shape)
display(gold_standard.head())
display(gold_standard['true_class'].value_counts())

(998, 3)

,trait,drug,true_class
0,DOID:10652,DB00843,1
1,DOID:10652,DB00674,1
2,DOID:10652,DB01043,1
3,DOID:10652,DB00989,1
4,DOID:10652,DB00810,0


true_class
1    755
0    243
Name: count, dtype: int64

# Load drug-disease predictions

In [6]:
current_prediction_files = list(OUTPUT_PREDICTIONS_DIR.glob('*.h5'))
display(len(current_prediction_files))

assert len(current_prediction_files) == N_PREDICTION_FILES_TOTAL, (
    f'Expected {N_PREDICTION_FILES_TOTAL} files (490), found {len(current_prediction_files)}.\n'
    f'Run 011-prediction-single_gene_based.ipynb and 011-prediction-gene_module_based.ipynb first.'
)

490

In [7]:
current_prediction_files[:5]

[PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Kidney_Cortex-data-top_250_genes-prediction_scores.h5'),
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Esophagus_Muscularis-projection-top_10_genes-prediction_scores.h5'),
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Brain_Caudate_basal_ganglia-projection-top_25_genes-prediction_scores.h5'),
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Esophagus_Gastroesophageal_Junction-projection-top_25_genes-prediction_scores.h5'),
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zs

In [8]:
def _get_tissue(data_value):
    """
    Extracts tissue name from the 'data' metadata field.
    Handles both raw (-data) and projected (-projection) suffixes.
    E.g.: 'spredixcan-mashr-zscores-Liver-data' -> 'Liver'
         'spredixcan-mashr-zscores-Liver-projection' -> 'Liver'
    """
    if data_value.endswith('-projection'):
        return data_value.split('spredixcan-mashr-zscores-')[1].split('-projection')[0]
    else:
        return data_value.split('spredixcan-mashr-zscores-')[1].split('-data')[0]

In [9]:
# Load all prediction files, rank scores, merge with gold standard
predictions = []

for f in tqdm(current_prediction_files, ncols=100):
    # Load DOID-mapped predictions and rank within the full DOID distribution
    prediction_data = pd.read_hdf(f, key='prediction')
    prediction_data['score'] = prediction_data['score'].rank()

    # Filter to gold-standard pairs
    prediction_data = pd.merge(
        prediction_data, gold_standard, on=['trait', 'drug'], how='inner'
    )
    prediction_data['trait'] = prediction_data['trait'].astype('category')
    prediction_data['drug'] = prediction_data['drug'].astype('category')

    # Read metadata
    metadata = pd.read_hdf(f, key='metadata')

    prediction_data = prediction_data.assign(method=metadata['method'].values[0])
    prediction_data['method'] = prediction_data['method'].astype('category')

    prediction_data = prediction_data.assign(n_top_genes=metadata['n_top_genes'].values[0])

    data_value = metadata['data'].values[0]
    prediction_data = prediction_data.assign(data=data_value)
    prediction_data['data'] = prediction_data['data'].astype('category')

    # Extract tissue name from data field
    prediction_data = prediction_data.assign(tissue=_get_tissue(data_value))

    predictions.append(prediction_data)

  0%|                                                                       | 0/490 [00:00<?, ?it/s]

  0%|▏                                                              | 1/490 [00:00<01:30,  5.40it/s]

  0%|▎                                                              | 2/490 [00:00<01:21,  5.96it/s]

  1%|▍                                                              | 3/490 [00:00<01:18,  6.17it/s]

  1%|▌                                                              | 4/490 [00:00<01:17,  6.28it/s]

...

 99%|████████████████████████████████████████████████████████████▋| 487/490 [01:14<00:00,  6.59it/s]

100%|████████████████████████████████████████████████████████████▊| 488/490 [01:14<00:00,  6.48it/s]

100%|████████████████████████████████████████████████████████████▉| 489/490 [01:15<00:00,  6.51it/s]

100%|█████████████████████████████████████████████████████████████| 490/490 [01:15<00:00,  6.54it/s]

100%|█████████████████████████████████████████████████████████████| 490/490 [01:15<00:00,  6.51it/s]

In [10]:
predictions = pd.concat(predictions, ignore_index=True)

In [11]:
display(predictions.shape)
display(predictions.head())

(335650, 8)

,trait,drug,score,true_class,method,n_top_genes,data,tissue
0,DOID:0050741,DB00215,200281.5,1,Gene-based,250.0,spredixcan-mashr-zscores-Kidney_Cortex-data,Kidney_Cortex
1,DOID:0050741,DB00704,319233.0,1,Gene-based,250.0,spredixcan-mashr-zscores-Kidney_Cortex-data,Kidney_Cortex
2,DOID:0050741,DB00822,343497.0,1,Gene-based,250.0,spredixcan-mashr-zscores-Kidney_Cortex-data,Kidney_Cortex
3,DOID:10283,DB00014,116486.0,1,Gene-based,250.0,spredixcan-mashr-zscores-Kidney_Cortex-data,Kidney_Cortex
4,DOID:10283,DB00175,148768.0,0,Gene-based,250.0,spredixcan-mashr-zscores-Kidney_Cortex-data,Kidney_Cortex


## Validation checks

In [12]:
assert not predictions.isna().any().any()

_tmp = predictions['method'].value_counts()
display(_tmp)

N_PREDICTIONS = predictions[['drug', 'trait']].drop_duplicates().shape[0]
display(f'Unique drug-disease pairs: {N_PREDICTIONS}')

assert _tmp.loc['Gene-based'] == N_TISSUES * N_THRESHOLDS * N_PREDICTIONS
assert _tmp.loc['Module-based'] == N_TISSUES * N_THRESHOLDS * N_PREDICTIONS

method
Gene-based      167825
Module-based    167825
Name: count, dtype: int64

'Unique drug-disease pairs: 685'

In [13]:
_tmp = predictions.groupby(['method', 'n_top_genes'], observed=True).count()
display(_tmp)
assert np.all(_tmp == N_TISSUES * N_PREDICTIONS)

trait   drug  score  true_class   data  tissue
method       n_top_genes                                                
Gene-based   -1.0         33565  33565  33565       33565  33565   33565
              50.0        33565  33565  33565       33565  33565   33565
              100.0       33565  33565  33565       33565  33565   33565
              250.0       33565  33565  33565       33565  33565   33565
              500.0       33565  33565  33565       33565  33565   33565
Module-based -1.0         33565  33565  33565       33565  33565   33565
              5.0         33565  33565  33565       33565  33565   33565
              10.0        33565  33565  33565       33565  33565   33565
              25.0        33565  33565  33565       33565  33565   33565
              50.0        33565  33565  33565       33565  33565   33565

## Save raw predictions

In [14]:
output_file = OUTPUT_DIR / 'predictions' / 'predictions_results.pkl'
display(output_file)
predictions.to_pickle(output_file)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/predictions_results.pkl')

# Aggregate predictions

1. Average ranks across n_top_genes thresholds (per trait, drug, method, tissue).
2. Take maximum across tissues (per trait, drug, method).

This matches the PhenoPlier aggregation exactly.

In [15]:
def _reduce_mean(x):
    return pd.Series({
        'score': x['score'].mean(),
        'true_class': x['true_class'].unique()[0],
    })


def _reduce_max(x):
    return pd.Series({
        'score': x['score'].max(),
        'true_class': x['true_class'].unique()[0],
    })

In [16]:
predictions_avg = (
    # Step 1: average across n_top_genes thresholds
    predictions
    .groupby(['trait', 'drug', 'method', 'tissue'], observed=True)
    .apply(_reduce_mean, include_groups=False)
    .dropna()
    # Step 2: take maximum across tissues
    .groupby(['trait', 'drug', 'method'], observed=True)
    .apply(_reduce_max, include_groups=False)
    .dropna()
    .sort_index()
    .reset_index()
)

In [17]:
display(predictions_avg.shape)
display(predictions_avg.head())

# Should have exactly 2 × N_PREDICTIONS rows (one per method)
assert predictions_avg.shape[0] == 2 * N_PREDICTIONS
assert predictions_avg.dropna().shape == predictions_avg.shape

(1370, 5)

,trait,drug,method,score,true_class
0,DOID:0050741,DB00215,Gene-based,316134.3,1.0
1,DOID:0050741,DB00215,Module-based,353878.5,1.0
2,DOID:0050741,DB00704,Gene-based,387103.6,1.0
3,DOID:0050741,DB00704,Module-based,393946.6,1.0
4,DOID:0050741,DB00822,Gene-based,409053.0,1.0


## Save aggregated predictions

In [18]:
output_file = OUTPUT_DIR / 'predictions' / 'predictions_results_aggregated.pkl'
display(output_file)
predictions_avg.to_pickle(output_file)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/predictions_results_aggregated.pkl')

# ROC performance

In [19]:
# AUROC per method and n_top_genes threshold
predictions.groupby(['method', 'tissue', 'n_top_genes'], observed=True).apply(
    lambda x: roc_auc_score(x['true_class'], x['score']), include_groups=False
).groupby(['method', 'n_top_genes'], observed=True).describe()

count      mean       std       min       25%  \
method       n_top_genes                                                  
Gene-based   -1.0          49.0  0.549364  0.023679  0.504653  0.536803   
              50.0         49.0  0.537231  0.023829  0.481431  0.522263   
              100.0        49.0  0.536558  0.018923  0.487655  0.526958   
              250.0        49.0  0.538607  0.023377  0.477309  0.521321   
              500.0        49.0  0.540913  0.022241  0.500984  0.523131   
Module-based -1.0          49.0  0.518573  0.022370  0.457572  0.504225   
              5.0          49.0  0.536431  0.028108  0.476808  0.520551   
              10.0         49.0  0.537466  0.031176  0.460348  0.523657   
              25.0         49.0  0.533242  0.031503  0.481455  0.504751   
              50.0         49.0  0.531007  0.025644  0.463662  0.512272   

                               50%       75%       max  
method       n_top_genes                                
Gene-based   -1.0         0.550034  0.564318  0.598705  
              50.0        0.535861  0.552089  0.607265  
              100.0       0.536619  0.550328  0.566788  
              250.0       0.539738  0.556467  0.583908  
              500.0       0.544030  0.556442  0.598693  
Module-based -1.0         0.519352  0.532278  0.573379  
              5.0         0.533464  0.551294  0.621707  
              10.0        0.539884  0.559304  0.613135  
              25.0        0.536387  0.550218  0.635709  
              50.0        0.529612  0.553630  0.586904

In [20]:
# Final AUROC using aggregated predictions
auroc_final = predictions_avg.groupby('method', observed=True).apply(
    lambda x: roc_auc_score(x['true_class'], x['score']), include_groups=False
).rename('AUROC')
display(auroc_final)

method
Gene-based      0.583382
Module-based    0.605737
Name: AUROC, dtype: float64

These are the final performance measures using AUROC.

# Precision-Recall performance

In [21]:
# Average precision per method and n_top_genes threshold
predictions.groupby(['method', 'tissue', 'n_top_genes'], observed=True).apply(
    lambda x: average_precision_score(x['true_class'], x['score']), include_groups=False
).groupby(['method', 'n_top_genes'], observed=True).describe()

count      mean       std       min       25%  \
method       n_top_genes                                                  
Gene-based   -1.0          49.0  0.823048  0.012367  0.791177  0.817139   
              50.0         49.0  0.819142  0.011775  0.793504  0.811550   
              100.0        49.0  0.819522  0.010505  0.782200  0.812821   
              250.0        49.0  0.820481  0.012213  0.788298  0.812494   
              500.0        49.0  0.820813  0.011201  0.801469  0.814246   
Module-based -1.0          49.0  0.806810  0.012836  0.776331  0.796252   
              5.0          49.0  0.812129  0.017114  0.776060  0.801291   
              10.0         49.0  0.813750  0.018278  0.775588  0.800520   
              25.0         49.0  0.813229  0.018512  0.778483  0.800619   
              50.0         49.0  0.811462  0.016060  0.766112  0.799672   

                               50%       75%       max  
method       n_top_genes                                
Gene-based   -1.0         0.822989  0.828268  0.845962  
              50.0        0.818870  0.827690  0.850680  
              100.0       0.820163  0.825928  0.837381  
              250.0       0.820298  0.829858  0.841266  
              500.0       0.819441  0.828770  0.849463  
Module-based -1.0         0.807951  0.817486  0.830619  
              5.0         0.809984  0.821604  0.861639  
              10.0        0.813314  0.829202  0.857331  
              25.0        0.813917  0.826922  0.865645  
              50.0        0.810987  0.822886  0.845370

In [22]:
# Final average precision using aggregated predictions
ap_final = predictions_avg.groupby('method', observed=True).apply(
    lambda x: average_precision_score(x['true_class'], x['score']), include_groups=False
).rename('AvgPrecision')
display(ap_final)

method
Gene-based      0.844942
Module-based    0.837282
Name: AvgPrecision, dtype: float64

These are the final performance measures using average precision.

# Summary

In [23]:
summary = pd.concat([auroc_final, ap_final], axis=1)
display(summary)

,AUROC,AvgPrecision
method,,
Gene-based,0.583382,0.844942
Module-based,0.605737,0.837282
